In [2]:
import pandas as pd
df = pd.read_csv("files/batch-3.csv")

In [3]:
print("Before:", len(df))

Before: 3326


In [4]:
terms = ["palestine", "palestinian", "gaza", "west bank"]

def has_term(text):
    t = str(text).lower()
    return any(term in t for term in terms)

df = df[df["full_text"].apply(has_term)]

print("After phrase filter:", len(df))


After phrase filter: 2830


In [5]:
df = df.drop_duplicates(subset=["screen_name", "full_text"], keep="first")

print("After author+text dedupe:", len(df))

After author+text dedupe: 2656


In [6]:
counts = df["screen_name"].value_counts()
print(counts.describe())
print()
print(counts.head(15))

count    1621.000000
mean        1.638495
std         3.535722
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max       108.000000
Name: count, dtype: float64

screen_name
QudsNen            108
JuniorM26048432     55
DmodosCutter        36
IhabHassane         21
ME_Observer_        20
WorldwideNNX        17
NodeWire            17
Aljarmaqnetnews     17
Marianhouk          16
FirstSquawk         15
PalTimesNews        14
palinfoen           14
anadoluagency       13
JANETGi59151282     13
sahatenglish        12
Name: count, dtype: int64


In [7]:

df = df.groupby("screen_name", group_keys=False).head(3)

print("After per-author cap (3):", len(df))

After per-author cap (3): 2091


In [11]:
import json

def extract_location(meta_str):
    try:
        m = json.loads(meta_str)
        return m["core"]["user_results"]["result"]["location"]["location"]
    except Exception:
        return None

def extract_hashtags(meta_str):
    try:
        m = json.loads(meta_str)
        tags = m.get("legacy", {}).get("entities", {}).get("hashtags", [])
        if not tags and m.get("note_tweet"):
            tags = (
                m["note_tweet"]["note_tweet_results"]["result"]
                .get("entity_set", {})
                .get("hashtags", [])
            )
        return [t["text"] for t in tags]
    except Exception:
        return []

df["profile_location"] = df["metadata"].apply(extract_location)
df["hashtags"] = df["metadata"].apply(extract_hashtags)

print(df["hashtags"].apply(len).gt(0).sum(), "rows with at least one hashtag")

144 rows with at least one hashtag


In [12]:
empty_loc = df["profile_location"].fillna("").str.strip().eq("")
print(empty_loc.sum(), "rows with no location")
print((~empty_loc).sum(), "rows with usable location")

0 rows with no location
1367 rows with usable location


In [13]:
print("Before:", len(df))

df = df[df["profile_location"].fillna("").str.strip() != ""]

print("After dropping empty locations:", len(df))

Before: 1367
After dropping empty locations: 1367


In [14]:
print(df["profile_location"].value_counts().head(30))
print()
print("Unique location strings:", df["profile_location"].nunique())

profile_location
Palestine                  33
United States              33
USA                        32
London                     23
London, England            23
Washington, DC             17
Canada                     15
United Kingdom             15
England, United Kingdom    13
Israel                     12
World                      12
California, USA            11
Global                     11
Florida, USA               10
New York                   10
Earth                      10
New York, NY               10
India                       9
UK                          8
Ireland                     8
New York, USA               8
Gaza                        8
Doha, Qatar                 7
Italy                       7
Nairobi, Kenya              6
Toronto, Ontario            6
New Delhi, India            6
Tennessee, USA              5
Tel Aviv, Israel            5
Los Angeles, CA             5
Name: count, dtype: int64

Unique location strings: 750


In [15]:
exclude_locations = ["Palestine", "Gaza", "World", "Global", "Earth"]

print("Before:", len(df))
df = df[~df["profile_location"].isin(exclude_locations)]
print("After excluding Palestine/Gaza/junk:", len(df))

Before: 1367
After excluding Palestine/Gaza/junk: 1293


In [16]:
variant_mask = df["profile_location"].str.lower().str.contains("palestine|gaza", na=False)
print(variant_mask.sum(), "remaining Palestine/Gaza-related variants")
print(df.loc[variant_mask, "profile_location"].value_counts())

16 remaining Palestine/Gaza-related variants
profile_location
Palestine                        5
Gaza, Palestine                  2
Jordan Valley, Palestine         1
Palestine, Lebanon & Jordan      1
Palestine, Gaza                  1
Ramallah, Palestine              1
    Gaza Strip, Palestine        1
Palestine, Gaza                  1
Gaza, Palestine / غزة، فلسطين    1
State of Palestine               1
Palestine 48                     1
Name: count, dtype: int64


In [17]:
print("Before:", len(df))
df = df[~df["profile_location"].str.lower().str.contains("palestine|gaza", na=False)]
print("After excluding all Palestine/Gaza variants:", len(df))

Before: 1293
After excluding all Palestine/Gaza variants: 1277


In [25]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="palestine_x_analysis")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def geocode_country(location):
    try:
        result = geocode(location, language="en", addressdetails=True)
        if result and "country" in result.raw.get("address", {}):
            return result.raw["address"]["country"]
        return None
    except Exception:
        return None

unique_locations = df["profile_location"].dropna().unique()
print("Unique locations to geocode:", len(unique_locations))

location_to_country = {}
for i, loc in enumerate(unique_locations):
    location_to_country[loc] = geocode_country(loc)
    if i % 50 == 0:
        print(f"{i}/{len(unique_locations)} done")

Unique locations to geocode: 734
0/734 done
50/734 done
100/734 done
150/734 done
200/734 done
250/734 done
300/734 done
350/734 done
400/734 done
450/734 done
500/734 done
550/734 done
600/734 done
650/734 done
700/734 done


In [26]:
df["country"] = df["profile_location"].map(location_to_country)

print("Mapped:", df["country"].notna().sum())
print("Unmapped:", df["country"].isna().sum())

Mapped: 1070
Unmapped: 207


In [28]:
print(df.loc[df["country"].isna(), "profile_location"].value_counts().head(30))

profile_location
Europe                            4
Ramallah / Denver, Colorado       3
Gauda Prime                       3
London, Mumbai, Singapore         3
Mobile:  0451655297               3
Your Twitter Feed                 3
Between Jerusalem & Tel Aviv      3
With Oppressed                    3
SoCal, from CT.                   3
Australia and Sag A               3
Best Damn Country on Earth-USA    3
Magical Earth 🌍                   3
SK, BC, FL, and in YOUR mind!     2
🇨🇦🇺🇸🏳️‍🌈                          2
Africa                            2
Praying for peace                 2
NJ,PA, FL                         2
Not where I want to be.           2
Datu Saudi-Ampatuan, Nagsasari    2
huna/hone/hina                    2
الناصرة فلسطين بلاد الشام         2
fundraising offline               2
ANTI-ZIONIST                      2
Mountains forests and glaciers    2
YVR + YHZ                         2
Zionist Regime of Canada          2
Right behind you                  1
U.N.DRIVE  

In [29]:
print("Before:", len(df))
df = df[df["country"].notna()]
print("After dropping unmapped locations:", len(df))

Before: 1277
After dropping unmapped locations: 1070


In [ ]:
final_columns = [
    "id", "created_at", "full_text", "screen_name", "name",
    "profile_location", "country", "hashtags",
    "favorite_count", "retweet_count", "bookmark_count",
    "quote_count", "reply_count", "views_count", "url"
]

processed = df[final_columns]

processed.to_csv("processed_posts.csv", index=False)

Saved: 1070 rows, 15 columns
